# Attesting a crosswalk decision with JWS

This notebook separates two questions that are easy to conflate:

1. **Should these records be linked?** Arche's deterministic crosswalk answers
   this from the records, comparator pack, thresholds, and evidence.
2. **Who attests to that decision, and did it change?** A JWS answers this after
   the decision is made.

The signing key does **not** alter a score, `match` or `review` decision,
evidence, or `decision_id`. It signs the result that already exists.

## Create one decision

The records below represent the same facility with a spelling difference. The
crosswalk gets no identifier, signing key, or issuer information. It only gets
names and coordinates.

In [1]:
from arche.resolve import crosswalk

registry = [{
    "id": "registry:001",
    "name": "Harbour View Health Centre",
    "lat": 6.5930,
    "lon": 3.3577,
}]
map_data = [{
    "id": "map:abc",
    "name": "Harbor View Health Centre",
    "lat": 6.5931,
    "lon": 3.3578,
}]

result = crosswalk(registry, map_data, entity="place")
edge = result["matches"][0]

print("decision:", edge["decision"])
print("score:", edge["score"])
print("decision_id:", edge["decision_id"])
print("evidence:", edge["evidence"])
print("pins:", result["pins"])

decision: match
score: 0.7737
decision_id: xwd:sha256:26e0ce96bd4df0e9965a8cce4fb2158e6348dbac3efd6a254ca23168a4146b26
evidence: {'name': 0.992, 'name_phrase': 'view health', 'name_phrase_rarity': 0.946, 'name_tftoken': 0.445, 'name_type': 1.0, 'geo': 0.995, 'distance_km': 0.02}
pins: {'engine': 'crosswalk.v1', 'comparators_sha256': '48e646d6895e9d1f', 'block': 'union', 'threshold': 0.7, 'review_margin': 0.15, 'distinctive_floor': 0.75, 'tf': 'shipped:place@sha256:c94f20a1c2dfba18+phrases@sha256:ed19b623d77407d3'}


`decision_id` starts with `xwd:sha256:`. It is a content hash, not a JWS. The
pins say which engine, comparator configuration, blocker, thresholds, and
frequency tables produced the decision.

## Sign the same result with two keys

These generated keys are for a demonstration only. In a real service, keep the
private key in a controlled signer or key-management system and distribute the
public key through a trusted channel.

In [2]:
from arche.resolve.reconcile import sign_edges
from arche.sign import generate_keypair

alice = generate_keypair()
bob = generate_keypair()

alice_attestation = sign_edges(
    result,
    private_key=alice.private_key,
    kid=alice.did_key,
)[0]
bob_attestation = sign_edges(
    result,
    private_key=bob.private_key,
    kid=bob.did_key,
)[0]

alice_jws = alice_attestation["jws"]
bob_jws = bob_attestation["jws"]

print("same decision ID:", alice_attestation["decision_id"] == bob_attestation["decision_id"])
print("different JWS values:", alice_jws != bob_jws)
print("compact JWS segments:", alice_jws.count(".") + 1)

same decision ID: True
different JWS values: True
compact JWS segments: 3


The decision ID remains the same because the decision did not change. The JWS
values differ because Alice and Bob used different private keys. A compact JWS
has three Base64URL segments: `header.payload.signature`.

## Verify before reading the payload

You can Base64URL-decode the header and payload for debugging, but do not trust
them until verification succeeds against a public key you already trust.

In [3]:
from arche.sign import verify

alice_check = verify(alice_jws, public_key=alice.public_key)
bob_check = verify(alice_jws, public_key=bob.public_key)

print("Alice key validates Alice JWS:", alice_check.valid, alice_check.trusted)
print("Bob key validates Alice JWS:", bob_check.valid, bob_check.trusted)

payload = alice_check.payload
print("signed schema:", payload["schema"])
print("signed decision ID:", payload["decision_id"])
print("payload decision ID matches result:", payload["decision_id"] == edge["decision_id"])
print("signed pins match result:", payload["pins"] == result["pins"])

Alice key validates Alice JWS: True True
Bob key validates Alice JWS: False False
signed schema: arche.crosswalk_edge.v1
signed decision ID: xwd:sha256:26e0ce96bd4df0e9965a8cce4fb2158e6348dbac3efd6a254ca23168a4146b26
payload decision ID matches result: True
signed pins match result: True


`valid` means the signature matches the supplied key. `trusted` means the key
came from the trusted `public_key` supplied to verification. Do not let a token
choose its own trust root in a production workflow.

## Detect tampering

Changing even one character in the signature makes the compact JWS fail
verification. The underlying decision is unaffected, but the attestation is no
longer usable.

In [4]:
header, payload_part, signature = alice_jws.split(".")
replacement = "A" if signature[-1] != "A" else "B"
tampered_jws = f"{header}.{payload_part}.{signature[:-1]}{replacement}"
tampered_check = verify(tampered_jws, public_key=alice.public_key)

print("tampered JWS validates:", tampered_check.valid)
print("verification error:", tampered_check.error)

tampered JWS validates: False
verification error: Ed25519 signature verification failed


## What would change a merge decision?

A key never changes the decision. Re-running `crosswalk` with changed source
records, comparators, thresholds, blocking policy, or frequency tables can
change the score, evidence, decision, pins, and `decision_id`. That new result
must be signed again.

In [5]:
repeat = crosswalk(registry, map_data, entity="place")
same_decision_id = repeat["matches"][0]["decision_id"] == edge["decision_id"]
print("same inputs give the same decision ID:", same_decision_id)

# A signer may attest to the repeat, but cannot make it a different merge.
repeat_attestation = sign_edges(
    repeat,
    private_key=alice.private_key,
    kid=alice.did_key,
)[0]
signing_preserves_id = repeat_attestation["decision_id"] == edge["decision_id"]
print("signing preserves the decision ID:", signing_preserves_id)

same inputs give the same decision ID: True
signing preserves the decision ID: True


## Keep and share

Store the crosswalk output, source hashes, decision pins, the compact JWS, and
the signer identity together. Share the public key through a trusted channel.
Never share a private key or use a notebook-generated key for production.